# Lambda Calibration — mini-tips → Run20

Finds the optimal `lam_thumb_tip / lam_tip / lam_pinch / lam_tip_rot / lam_dr`
so that the mini-tips metric best approximates Run20's ahg metric
(`S_k = W_R*D_R + W_JOINTS*D_joints + W_AHG*D_ahg`).

Run20 weights: `W_R=1.0, W_JOINTS=1.2, W_AHG=0.07`.

Method: mini-tips S_k is **linear** in the lambdas → least squares on N pose pairs.
R² tells you how well mini-tips CAN approximate Run20 (structural ceiling).

In [ ]:
import os
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT  = Path('/content/drive/MyDrive/AIST-hand')
GITHUB_TOKEN = ''   # set if private repo
GITHUB_USER  = 'isapedraza'
REPO_NAME    = 'AIST-hand'

In [ ]:
REPO_DIR = f'/content/{REPO_NAME}'
BRANCH   = 'main'

if not os.path.exists(REPO_DIR):
    clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    os.system(f'git clone {clone_url} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} fetch origin')

os.system(f'git -C {REPO_DIR} checkout {BRANCH}')
os.system(f'git -C {REPO_DIR} pull origin {BRANCH}')
print(f'Branch: {BRANCH}')

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'setuptools-scm'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-kinematics'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e',
                f'{REPO_DIR}/models/latent-retargeting'], check=True)
print('Done.')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
# Path to shadow 1M_dong NPZ on Drive (must have quats/chain/tips keys)
VALID_POSES_PATH = DRIVE_ROOT / 'shadow_1M_dong.npz'

N_SAMPLE = 20_000    # → 10k pairs (split a/b)

# Run20 S_k weights (from commit e330f1d)
W_R      = 1.0
W_JOINTS = 1.2
W_AHG    = 0.07   # negligible — included for completeness

# Current lambda baseline (for R² comparison)
CURRENT = dict(lam_thumb_tip=1.0, lam_tip=1.0, lam_pinch=1.3, lam_tip_rot=0.3, lam_dr=1.2)

In [ ]:
import numpy as np
import torch

# Load DONG_CACHE directly — no FK needed, data already pre-computed
print(f'Loading {VALID_POSES_PATH} ...')
with np.load(VALID_POSES_PATH) as data:
    required = ('quats', 'chain', 'tips', 'joint_labels', 'tip_labels')
    missing  = [k for k in required if k not in data.files]
    if missing:
        raise ValueError(f'NPZ missing keys: {missing}. Need DONG_CACHE format.')
    quats       = torch.from_numpy(data['quats']).float()        # [N, Jk, 4]
    chain       = torch.from_numpy(data['chain']).float()        # [N, F, 4, 3]
    tips        = torch.from_numpy(data['tips']).float()         # [N, F, 3]
    joint_labels = [str(s) for s in data['joint_labels']]        # e.g. ['thumb_mcp',...]
    tip_labels   = [str(s) for s in data['tip_labels']]          # e.g. ['thumb','index',...]

N_total = tips.shape[0]
print(f'Loaded: {N_total:,} poses | joints={len(joint_labels)} | fingers={tip_labels}')
print(f'quats={tuple(quats.shape)}  chain={tuple(chain.shape)}  tips={tuple(tips.shape)}')

In [ ]:
# Sample N_SAMPLE poses and split into N_pairs pairs
torch.manual_seed(42)
idx  = torch.randperm(N_total)[:N_SAMPLE]
N_pairs = N_SAMPLE // 2

tips_a  = tips[idx[:N_pairs]]    # [N_pairs, F, 3]
tips_b  = tips[idx[N_pairs:]]    # [N_pairs, F, 3]
chain_a = chain[idx[:N_pairs]]   # [N_pairs, F, 4, 3]
chain_b = chain[idx[N_pairs:]]   # [N_pairs, F, 4, 3]
quats_a = quats[idx[:N_pairs]]   # [N_pairs, Jk, 4]
quats_b = quats[idx[N_pairs:]]   # [N_pairs, Jk, 4]

print(f'{N_pairs:,} pairs ready')

In [ ]:
import sys
sys.path.insert(0, f'{REPO_DIR}/models/latent-retargeting/src')
from cross_emb.training.losses import _sk_w, _sk_wj, _ahg

FINGER_ORDER = ['thumb', 'index', 'middle', 'ring', 'pinky']
SEG_ORDER    = ['mcp', 'pip', 'dip', 'tip']

# Design matrix A [N_pairs*5, 5] and target b [N_pairs*5]
# Lambda columns: [lam_thumb_tip, lam_tip, lam_pinch, lam_tip_rot, lam_dr]
A_rows, b_rows = [], []

thumb_fi = tip_labels.index('thumb')

for finger in FINGER_ORDER:
    fi = tip_labels.index(finger)

    # ── mini-tips features ────────────────────────────────────────────────────
    # d_tip: ||tip_a - tip_b||²
    d_tip = ((tips_a[:, fi] - tips_b[:, fi]) ** 2).sum(-1)           # [N_pairs]

    # d_pinch: ||(tip_a - thumb_a) - (tip_b - thumb_b)||²  (index/middle/ring only)
    if finger in ('index', 'middle', 'ring'):
        g_a     = tips_a[:, fi] - tips_a[:, thumb_fi]
        g_b     = tips_b[:, fi] - tips_b[:, thumb_fi]
        d_pinch = ((g_a - g_b) ** 2).sum(-1)
    else:
        d_pinch = torch.zeros(N_pairs)

    # d_tip_rot: ||r_a_hat - r_b_hat||²  where r = TIP - DIP (last chain segment)
    r_a     = chain_a[:, fi, 3] - chain_a[:, fi, 2]                   # [N_pairs, 3]
    r_b     = chain_b[:, fi, 3] - chain_b[:, fi, 2]
    r_a_hat = r_a / r_a.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    r_b_hat = r_b / r_b.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    d_tip_rot = ((r_a_hat - r_b_hat) ** 2).sum(-1)

    # D_R: Σ_j w_j * (1 - (q_a_j · q_b_j)²)
    seg_labels = [f'{finger}_{seg}' for seg in SEG_ORDER]
    jidx = [joint_labels.index(l) for l in seg_labels if l in joint_labels]
    w_dr = torch.tensor(_sk_w[finger][:len(jidx)])                    # [n_joints]
    w_dr = w_dr / w_dr.sum().clamp(min=1e-8)
    q_a_f = quats_a[:, jidx]                                          # [N_pairs, n_joints, 4]
    q_b_f = quats_b[:, jidx]
    per_jdr = 1.0 - (q_a_f * q_b_f).sum(-1) ** 2                     # [N_pairs, n_joints]
    D_R = (w_dr * per_jdr).sum(-1)                                    # [N_pairs]

    # ── Run20 ahg target ─────────────────────────────────────────────────────
    # D_joints: Σ_j w_j * ||chain_a[j] - chain_b[j]||
    w_joints_f  = torch.tensor(_sk_wj[finger])                        # [4]
    chain_diff  = (chain_a[:, fi] - chain_b[:, fi]).norm(dim=-1)      # [N_pairs, 4]
    D_joints    = (w_joints_f * chain_diff).sum(-1)                   # [N_pairs]

    # D_ahg per finger (expensive but W_AHG=0.07 so near-zero contribution)
    # Using the full _ahg function over the single finger chain
    c_a_f = chain_a[:, fi:fi+1]   # [N_pairs, 1, 4, 3]  — kept as 1-finger for _ahg
    c_b_f = chain_b[:, fi:fi+1]
    D_ahg = _ahg(c_a_f, c_b_f)                                        # [N_pairs]

    target = W_R * D_R + W_JOINTS * D_joints + W_AHG * D_ahg

    # ── design matrix row ────────────────────────────────────────────────────
    # Columns: [lam_thumb_tip, lam_tip, lam_pinch, lam_tip_rot, lam_dr]
    zeros = torch.zeros(N_pairs)
    if finger == 'thumb':
        row = torch.stack([d_tip, zeros, zeros, d_tip_rot, D_R], dim=1)
    else:
        row = torch.stack([zeros, d_tip, d_pinch, d_tip_rot, D_R], dim=1)

    A_rows.append(row)
    b_rows.append(target)
    print(f'{finger:6s}  D_R={D_R.mean():.4f}  D_joints={D_joints.mean():.4f}  target_mean={target.mean():.4f}')

A = torch.cat(A_rows, dim=0).numpy()   # [N_pairs*5, 5]
b = torch.cat(b_rows, dim=0).numpy()   # [N_pairs*5]
print(f'\nA={A.shape}  b={b.shape}')

In [ ]:
from numpy.linalg import lstsq

lam_names = ['lam_thumb_tip', 'lam_tip', 'lam_pinch', 'lam_tip_rot', 'lam_dr']

lambda_star, _, rank, _ = lstsq(A, b, rcond=None)

b_pred_star = A @ lambda_star
ss_tot  = ((b - b.mean()) ** 2).sum()

def r2(pred):
    return 1 - ((b - pred) ** 2).sum() / ss_tot

R2_opt = r2(b_pred_star)
b_pred_cur = A @ np.array([CURRENT[n] for n in lam_names])
R2_cur = r2(b_pred_cur)

print('=' * 55)
print(f'  Rank = {rank}   (5 = full rank, good)')
print('=' * 55)
print(f'  Current lambdas  R² = {R2_cur:.4f}')
print(f'  Optimal lambdas  R² = {R2_opt:.4f}')
print('=' * 55)
print()
print(f'{"name":20s}  {"current":>10s}  {"optimal":>10s}')
print('-' * 45)
for name, cur_val, opt_val in zip(lam_names, [CURRENT[n] for n in lam_names], lambda_star):
    arrow = '  ↑' if opt_val > cur_val + 0.05 else ('  ↓' if opt_val < cur_val - 0.05 else '')
    print(f'{name:20s}  {cur_val:10.4f}  {opt_val:10.4f}{arrow}')

In [ ]:
# Per-finger R² breakdown
print('Per-finger R² (optimal lambdas):')
for i, finger in enumerate(FINGER_ORDER):
    sl  = slice(i * N_pairs, (i + 1) * N_pairs)
    b_f = b[sl]
    p_f = b_pred_star[sl]
    ss_r = ((b_f - p_f) ** 2).sum()
    ss_t = ((b_f - b_f.mean()) ** 2).sum()
    print(f'  {finger:6s}: R² = {1 - ss_r/ss_t:.4f}')

print()
print('Interpretation:')
print('  R² > 0.90  -> mini-tips can approximate Run20; use optimal lambdas.')
print('  R² 0.7-0.9 -> partial; improvement likely but structural gap remains.')
print('  R² < 0.70  -> structural gap; D_joints signal not recoverable from tips alone.')

In [ ]:
# Feature scale diagnostics — understand relative magnitude of each term
print('Mean feature magnitudes per finger (helps interpret lambda scale):')
feat_names = ['d_tip', 'zeros', 'd_pinch', 'd_tip_rot', 'D_R']
for i, finger in enumerate(FINGER_ORDER):
    sl = slice(i * N_pairs, (i + 1) * N_pairs)
    row = A[sl]
    means = row.mean(axis=0)
    print(f'  {finger:6s}: ' + '  '.join(f'{n}={v:.4f}' for n, v in zip(lam_names, means)))

print()
print('Mean target (Run20 S_k) per finger:')
for i, finger in enumerate(FINGER_ORDER):
    sl = slice(i * N_pairs, (i + 1) * N_pairs)
    print(f'  {finger:6s}: {b[sl].mean():.4f}')